In [ ]:
import subprocess

llvm_path = "/home/dxic/llvmlearn/llvm-project/llvm/build/bin/"

def countloop(compiler, flags, input_file):
    cmd_list = [
        llvm_path + compiler,
        *flags,
        input_file,
        "-O3",
        "-Xclang",
        "-disable-llvm-passes",
        "-S",
        "-emit-llvm",
        "-o",
        "-",
    ]
    result = subprocess.run(
        # ' '.join(cmd_list), # handle backslash-escapes
        # shell=True,
        cmd_list,
        stdout = subprocess.PIPE,
        encoding="utf-8",
        check = True
    )

    init_ir = result.stdout
    cmd_list =[
        "/home/dxic/llvmtest/llvm-countloop/llvm-countloops",
    ]
    result = subprocess.run(cmd_list, input=init_ir, encoding= "utf-8", stdout = subprocess.PIPE, check=True)
    loopinfo = result.stdout.splitlines()
    funcloop = {}
    for line in loopinfo[:-1]:
        func, nloop = line.split(":")
        funcloop[func] = int(nloop)
    funcloop[" total"]=int(loopinfo[-1])
    return funcloop, len(init_ir)


In [ ]:
import json
import os
from concurrent.futures import ThreadPoolExecutor

nc, ncpp, nfunc0loop, nfunc1loop, nfunc2loop, snloop = 0, 0, 0, 0, 0, 0

def _task(cmd_split, dir, file, testname, idx):
    global nc, ncpp, nfunc0loop, nfunc1loop, nfunc2loop, snloop

    # if "/llvm-test-suite/SingleSource/" not in file:
    #     raise ValueError(f"file not in llvm-test-suite/SingleSource/{testname}{idx}")
    # if "/openssl/test/" in file:
    #     raise ValueError(f"file in openssl/test/{testname}{idx}")

    # Bear may record Clang's internal cc1 invocation as well as the driver command.
    # countloop must re-invoke the driver, so the internal form is not usable here.
    if "-cc1" in cmd_split:
        raise ValueError(f"-cc1 flags {file} {testname}{idx}")

    if not cmd_split[0].endswith("clang") and not cmd_split[0].endswith("clang++"):
        raise ValueError(f"not clang or clang++ {file} {testname}{idx}")
    compiler = cmd_split[0].rpartition("/")[-1]
    cmd_split = cmd_split[1:]
    if  compiler == "clang":
        nc += 1
    elif compiler == "clang++":
        ncpp += 1

    if not file.endswith(".c"):
        raise ValueError(f"not c file {file} {testname}{idx}")
    workdir = os.path.abspath(dir)
    command_file = os.path.abspath(os.path.join(workdir, cmd_split[-1]))
    if os.path.normcase(command_file) != os.path.normcase(os.path.abspath(file)):
        raise ValueError(f"command file does not match file: {command_file} != {file} {testname}{idx}")
    cmd_split = cmd_split[:-1]

    flag, i = [], 0
    while i < len(cmd_split):
        arg = cmd_split[i]
        i+=1
        if arg in ["-g", "-O2", "-O3", "-O1", "-O0", "-Os","-O", "-c", ]:
            continue
        if arg == "-o":
            i+=1
            continue
        if arg in ["-rpath", "-L", "-Xlinker"]:
            i+=1
            continue
        if arg.startswith(("-Wl,", "-l")) or arg.endswith((".o", ".obj", ".a", ".so", ".dylib")):
            continue
        if arg.startswith("-W") and not arg.startswith("-Wno") and not arg.startswith("-Wa,") and not arg.startswith("-Wp,") and not arg.startswith("-Wl,"):
            continue

        if arg == "-I":
            if i >= len(cmd_split):
                raise ValueError(f"missing path after -I {file} {testname}{idx}")
            include_path = cmd_split[i]
            i += 1
            arg = f"-I{os.path.abspath(os.path.join(workdir, include_path))}"
        elif arg.startswith("-I"):
            arg = f"-I{os.path.abspath(os.path.join(workdir, arg[2:]))}"

        flag.append(arg)
    if '-w' not in flag:
        flag.append('-w')
    
    try:
        funcloop, irsize = countloop(compiler, flag, file)
    except:
        raise ValueError(f"countloop failed {file} {testname}{idx}")
    # if funcloop[" total"]<=0:
    #     raise ValueError(f"no loop{testname}{idx}")
    snloop += funcloop[" total"]
    for func, nloop in funcloop.items():
        if func==" total":
            continue

        if nloop == 0:
            nfunc0loop += 1
        elif nloop == 1:
            nfunc1loop += 1
        else:
            nfunc2loop += 1
    
    testcase = {
        "compiler" : compiler,
        "flags" : flag,
        "input_file" : file,
        "id" : f"{testname}{idx}"
    }

    return testcase

def extract_tcases_from_bear(json_file,  testname):
    with open(json_file, 'r') as f:
        data = json.load(f)
    global nc, ncpp, nfunc0loop, nfunc1loop, nfunc2loop, snloop
    test, nc, ncpp, nfunc0loop, nfunc1loop, nfunc2loop, snloop = [], 0, 0, 0, 0, 0, 0
    
    with ThreadPoolExecutor(max_workers=10) as executor:
        futures= []
        for idx, entry in enumerate(data):
            cmd_split = entry["arguments"]
            dir = entry["directory"]
            file = entry["file"]
            # output = entry.get("output")

            future = executor.submit(_task, cmd_split, dir, file, testname, idx)
            futures.append(future)
        
        for future in futures:
            try:
                testcase = future.result()
                if testcase is not None:
                    test.append(testcase)
            except Exception as e:
                print(e)
                # if "countloop" in str(e):
                #     print(e)
                #     raise e
    
    print( nc, ncpp)
    print(len(test), nfunc0loop,nfunc1loop, nfunc2loop, snloop)


    # with open(output_file, 'w') as f:
    #     json.dump(test, f, indent=2, sort_keys=False)
    
    return test


In [ ]:
coreutils = extract_tcases_from_bear("./coreutils/compile_commands.json", "coreutils")
with open("coreutils260808.json", 'w') as f:
    json.dump(coreutils, f, indent=2, sort_keys=False)


In [ ]:
openssl = extract_tcases_from_bear("./openssl/compile_commands.json", "openssl")
with open("openssl260808.json", 'w') as f:
    json.dump(openssl, f, indent=2, sort_keys=False)


In [ ]:
ffmpeg = extract_tcases_from_bear("./ffmpeg/compile_commands.json", "ffmpeg")

with open("ffmpeg260808.json", 'w') as f:
    json.dump(ffmpeg, f, indent=2, sort_keys=False)

In [ ]:
curl = extract_tcases_from_bear("./curl/compile_commands.json", "curl")

with open("curl260808.json", 'w') as f:
    json.dump(curl, f, indent=2, sort_keys=False)

In [ ]:
zlib = extract_tcases_from_bear("./zlib/compile_commands.json", "zlib")

with open("zlib260808.json", 'w') as f:
    json.dump(zlib, f, indent=2, sort_keys=False)

In [ ]:
# sqlite3.c is too big to compile in parallel, replace it with multi c files (--disable-amalgamation)
sqlite = extract_tcases_from_bear("./sqlite/compile_commands.json", "sqlite")
# sqlite = [testcase for testcase in sqlite if not testcase["input_file"].endswith("sqlite3.c")]
with open("sqlite260808.json", 'w') as f:
    json.dump(sqlite, f, indent=2, sort_keys=False)



In [ ]:
testsuite = extract_tcases_from_bear("./llvm-test-suite/compile_commands.json", "testsuite")

with open("testsuite260808.json", 'w') as f:
    json.dump(testsuite, f, indent=2, sort_keys=False)